# Computer lab 3: duality

**Mathematical Optimization, VU Amsterdam, 2026–27.**

Derive each dual before implementing it. Keep track of the objective sense,
constraint orientation and variable domains. Check primal and dual feasibility
as well as their objective values, and explain what the multipliers mean.

## Setup

Run the next two cells before starting. They prepare HiGHS for Exercises 1 and
3–5 and Ipopt for Exercise 2.
If asked to restart the session, restart and run Setup again. See the
[course setup instructions](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local use and troubleshooting.

In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import time
import types
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

SETUP_COMMIT = '425c3f0af23e082c276df85fcde556538cc2abef'
SETUP_SHA256 = "f742478c6eaf5c5db82f4ce69b47a2adcf6f920765d88c3e7eb1b1ab72e66d28"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    for attempt in range(3):
        try:
            with urlopen(setup_location, timeout=30) as response:
                setup_source = response.read()
            break
        except (URLError, TimeoutError, ConnectionError) as error:
            if (isinstance(error, HTTPError) and error.code not in (408, 429, 500, 502, 503, 504)) or attempt == 2:
                raise
            time.sleep(0.5 * (attempt + 1))
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if local_repo and lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=3, check=False, verbose=True)
solve_and_load = lab_setup.solve_and_load
format_number = lambda value, digits=4: lab_setup.format_number(value, digits)
lp_solver, nlp_solver = lab.lp_solver, lab.nlp_solver
dual_display = lab.helpers['tutorial3']
add_dual_variables = dual_display.add_dual_variables
display_dual_variables = dual_display.display_dual_variables

In [ ]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

Useful references are the MO-book's [dual of the BIM production problem](https://mobook.github.io/MO-book/notebooks/02/06-bim-dual.html)
and the [Pyomo suffix documentation](https://pyomo.readthedocs.io/en/6.10.1/explanation/modeling/math_programming/suffixes.html).
The BIM example is a maximization primal; the two primals below are minimization
problems, so check their signs explicitly.

## Imported row duals and Lagrangian multipliers

For a **minimization primal**, the solvers used here report a raw RHS dual $d_i$
for each constraint row:

| Row orientation | Sign of raw RHS dual |
|:---|:---|
| $a_i^\top x\ge b_i$ | $d_i\ge0$ |
| $a_i^\top x\le b_i$ | $d_i\le0$ |
| $a_i^\top x=b_i$ | $d_i$ unrestricted |

Where the optimal-value function is differentiable, $d_i$ is its sensitivity to
increasing the RHS $b_i$ of that row. Here "RHS" means the row's bound $b_i$,
wherever Pyomo prints it. Negating a row changes the reported dual sign.
Exercise 1 uses LP dual variables $y_i$ in this RHS convention.

In Exercise 2 we write every inequality as $g_i(x)\le0$ and use
$L(x,\lambda)=f(x)+\sum_i\lambda_i g_i(x)$ with $\lambda_i\ge0$.
For these upper-bound rows, **$\lambda_i=-d_i$**. Keep the two sign conventions
separate when comparing imported values with a hand-derived dual.

Attach `model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)` **before solving**.
After a successful solve, read `model.dual[model.row[i]]` for an indexed row.
The imported course function `add_dual_variables(model)` also attaches this suffix.

As in weeks 1–2, `solve_and_load` checks termination before loading a solution.
Pass the solver explicitly and then evaluate with `pyo.value(...)`. Ipopt checks
local optimality conditions; convexity and the primal–dual certificates below
justify global conclusions here.

## Exercise 1: a linear primal and its dual

Consider the LP

$$
\begin{aligned}
\min\quad &x_1-2x_2+4x_3\\
\text{s.t.}\quad &-x_1+3x_2=3 &&(1),\\
&2x_1-x_2+3x_3\ge-5 &&(2),\\
&x_3\le7 &&(3),\\
&x_1+x_2-x_3\le11 &&(4),\\
&x_1,x_2,x_3\in\mathbb{R}.
\end{aligned}
$$

**(a)** Implement and solve the primal with HiGHS. Keep all three variables free
and attach a dual suffix before solving.

**(b)** Derive the dual by hand. Explain the sign of each dual variable and why
the dual constraints are equalities. Implement and solve the dual with HiGHS.

**(c)** Read the imported row duals directly from the primal suffix and compare
them with your explicit dual solution. You can also use
`display_dual_variables(model)` to display the rows and their raw duals.
Does strong duality hold, and could you have predicted it? Interpret the signs
of the nonzero RHS duals in this example.

In [ ]:
linear_rhs = {1: 3, 2: -5, 3: 7, 4: 11}

In [ ]:
# Build the primal with real-valued variables and the original row orientations.
# Attach a dual suffix before solving with solve_and_load(model, lp_solver).
# Report the decisions and objective value.

In [ ]:
# Derive the dual and its variable domains from the original LP.
# Implement the sign restrictions as variable domains and solve with HiGHS.
# Report the four dual variables and objective value.

In [ ]:
# Read model.dual[model.row[i]] after the primal solve and compare with your dual.
# Call display_dual_variables(model) to see each coded row and its imported sign.
# Check the objective gap and explain strong duality and the nonzero dual signs.

## Exercise 2: a convex quadratic primal and its Lagrangian dual

Consider

$$
\begin{aligned}
\min\quad &f(x)=x_1^2+x_2^2+x_3^2\\
\text{s.t.}\quad &g_1(x)=2x_1+x_2-5\le0,\\
&g_2(x)=x_1+x_3-2\le0,\\
&g_3(x)=1-x_1\le0,\\
&g_4(x)=2-x_2\le0,\\
&g_5(x)=-x_3\le0,\qquad x\in\mathbb{R}^3.
\end{aligned}
$$

**(a)** Explain convexity, implement the primal and solve it with Ipopt. Keep the
five inequalities in the stated $g_i(x)\le0$ orientation. Import the raw row duals
and convert them to the nonnegative multipliers in $L=f+\sum_i\lambda_i g_i$.

**(b)** Derive the Lagrangian dual by minimizing $L$ over the unrestricted vector
$x$. Implement and solve the resulting dual with Ipopt. Completing the square
gives a compact expression for the dual function.

**(c)** Check strong duality and explain whether it was predictable. Verify
complementary slackness and identify the binding constraints. Can a binding
constraint have a zero multiplier in this example?

In [ ]:
# Matrix form of the five stated inequalities: g(x) = A @ x - b <= 0.
quadratic_A = np.array([
    [2, 1, 0],
    [1, 0, 1],
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, -1],
], dtype=float)
quadratic_b = np.array([5, 2, -1, -2, 0], dtype=float)

In [ ]:
# Build the quadratic primal using g_i(x)<=0 exactly as written.
# Keep all five rows explicit and attach a dual suffix before solving with Ipopt.
# Read each indexed row's raw dual and form lambda_i = -raw_dual_i.
# Report the decisions, objective, raw duals and converted multipliers.

In [ ]:
# Complete the square in x to derive ell(lambda), with lambda>=0.
# Build an unsolved model maximizing the concave quadratic dual function.
# Solve with Ipopt and compare its multipliers and objective with part (a).

In [ ]:
# Explain strong duality, using convexity and a suitable Slater argument.
# Check primal/dual feasibility, stationarity and complementary slackness.
# Identify the binding rows and determine whether each must have a positive multiplier.
# Retain raw numerical values when checking residuals and the objective gap.

## Exercise 3: fixed decisions, prices and feasibility limits

A production unit has capacity 6 and an opening charge of 10. Its output $f$
costs 1 per unit; backup output $g$ costs 4 per unit. For demand $d\ge0$, consider

$$
\begin{aligned}
\min\quad &10z+f+4g\\
\text{s.t.}\quad &f+g=d,\\
&f\le6z,\qquad f,g\ge0,\qquad z\in\{0,1\}.
\end{aligned}
$$

The supplied builder returns an unsolved model. Use `fixed_design=None` for the
MILP, or 0 or 1 to fix only $z$; $f$ and $g$ remain variables. Setting
`backup=False` fixes $g=0$. Build a fresh model for each experiment.

**(a)** Solve the MILP at $d=3,4,6,7$. Fix $z$ to the design chosen at $d=3$,
apply `relax_integrality(model)`, and attach a dual suffix before solving the
resulting LP at $d=3$ and $d=4$. Compare the demand price with the finite cost
change under this fixed design and with the change when redesign is allowed.
Why are these different comparisons? Checks: the MILP values at 3 and 4 are
12 and 14; the fixed design costs 16 at 4. The MILP value at 7 is 20.

**(b)** Fix $z=1$, as chosen at $d=6$ and $d=7$. Derive its value function
$V_1(d)$ by considering whether $d\le6$. At $d=6$, compute the left and right
differences using $h=10^{-3}$, and compare them with the imported demand dual
$\pi$. Repeat the dual calculation at $d=7$. The capacity rent $\eta$ is the
negative of the raw upper-row dual `model.dual[model.capacity]`. Determine the possible pairs
$(\pi,\eta)$ at the kink. Must a returned dual equal one of the two slopes?
Must binding capacity have positive rent?

**(c)** At $d=5$ with $z=1$ fixed, compare the LP with an equivalent model whose
balance is $-f-g=-d$. Explain the change in the raw equality dual and why the
sensitivity to physical demand is unchanged.

**(d)** Remove backup and keep $z=1$. Solve at $d=6$ and $d=6.1$ using `solve`.
Report both statuses. Compute a right difference only if both solves are optimal;
otherwise report `None`. Interpret the value beyond the feasibility limit.
Never read variable values or duals from an infeasible solve.

**(e), optional.** Let $C\ge0$ be available capacity and exclude the opening
charge. Derive the operating cost $W(C;d)$, then compare the saving from opening
capacity 6 with the charge 10. Find the demand where the two designs tie.
At $C=0,d=3$, derive all optimal demand and capacity multipliers. Compare the
capacity rents with the right derivative of $W$ at zero.

`solve(model)` returns a dictionary containing `status` and `objective`; the
latter is `None` if no feasible solution was loaded. Fixed binary domains still
prevent dual loading: fix the design, relax integrality, then attach the suffix
and solve. Relaxing an unfixed design would give a different problem.

In [ ]:
solve = lab_setup.solve
relax_integrality = lab_setup.relax_integrality


def build_dispatch(d, fixed_design=None, backup=True):
    model = pyo.ConcreteModel(name="Production and backup")
    model.z = pyo.Var(domain=pyo.Binary)
    model.f = pyo.Var(domain=pyo.NonNegativeReals)
    model.g = pyo.Var(domain=pyo.NonNegativeReals)
    model.balance = pyo.Constraint(expr=model.f + model.g == d)
    model.capacity = pyo.Constraint(expr=model.f <= 6 * model.z)
    model.objective = pyo.Objective(expr=10 * model.z + model.f + 4 * model.g)
    if fixed_design is not None:
        model.z.fix(fixed_design)
    if not backup:
        model.g.fix(0)
    return model

In [ ]:
# Solve the four MILPs, check status, and report z, f, g and cost.
# Rebuild at d=3 and d=4 with only the design chosen at d=3 fixed.
# Relax integrality, attach the dual suffix, and solve each LP with HiGHS.
# Compare the demand dual with both finite cost changes.

In [ ]:
# Derive V_1(d) and its one-sided slopes at d=6.
# Solve fresh fixed-design LPs at d=6 and d=7; report pi and eta=-raw_capacity_dual.
# Rebuild at 6-h and 6+h to compute the two differences, using h=1e-3.
# Derive the dual-feasible interval at the kink and check the returned pair.

In [ ]:
# At d=5 and fixed z=1, solve the original and negated balance versions.
# Compare their objectives and raw duals, then account for the RHS sign.

In [ ]:
# Build fresh models without backup at d=6 and d=6.1, with z=1 fixed.
# Use solve(model) and report statuses before considering objective values.
# Report None for the numerical difference if either solve is not optimal.

## Exercise 4: maximum flow, competing cuts and integrality

Use Network 2 from week 1, supplied again below, with source 0 and sink 10.
We now seek maximum throughput: replace the original supplies and demands by
outgoing minus incoming flow equal to $v$ at the source, $-v$ at the sink and
zero at every intermediate node. Arc capacities are unchanged; costs are unused.

**(a)** Implement `build_max_flow(network, source, sink)` with continuous
nonnegative flows and throughput $v$, maximizing $v$. Keep capacity limits as
explicit rows and attach a dual suffix. Solve with HiGHS and report the flows,
throughput and capacity duals. Check: the maximum throughput is 15.

**(b)** The residual graph has forward capacity $u_{ij}-f_{ij}$ and backward
capacity $f_{ij}$ on each arc. The supplied function searches this graph.
Call it from the source to obtain a source side
$S_1$. Call it from the sink with `reverse=True` to find nodes that can reach the
sink; take their complement to obtain another source side $S_2$. For each side,
list the original arcs leaving it and sum their capacities directly from the
data. Compare these bounds with the throughput and with the nonzero capacity
duals. Explain the connection to strong LP duality. Does one returned dual
identify every minimum cut?

**(c)** Predict the new throughput when increasing the capacity of $(0,1)$ by
1; of both $(0,1)$ and $(1,2)$ by 1; and of both by 10. Solve a fresh model for
each case, always starting from the original data. Find a cut explaining the
last result. A saturated arc alone need not be the only bottleneck.

**(d)** Inspect the flows in (a) and recall the continuous flow models in week 1.
State the integrality theorem for a directed node–arc incidence matrix, including
its assumptions on balances and bounds. Then solve the continuous LP

$$
\max\ x_1+x_2+x_3\quad\text{s.t.}\quad
x_1+x_2\le1,\quad x_2+x_3\le1,\quad x_1+x_3\le1,\quad x\ge0.
$$

Show that its maximum is attained at a unique fractional vertex and find the
best integral value by hand. Minimize the same objective over the same feasible
set. Does an integral optimizer for this second objective establish that the
polytope is integral? Explain why the directed-incidence theorem does not apply
to this triangle matrix, whose columns each contain two $+1$ entries.

In [ ]:
flow_network = {
    "nodes": list(range(11)),
    "edges": {
        (0, 1): {"u": 15}, (1, 2): {"u": 15}, (2, 9): {"u": 10},
        (2, 3): {"u": 15}, (3, 4): {"u": 10}, (3, 5): {"u": 10},
        (4, 7): {"u": 4}, (4, 8): {"u": 5}, (5, 8): {"u": 5},
        (5, 9): {"u": 6}, (5, 10): {"u": 5}, (8, 6): {"u": 8},
        (6, 10): {"u": 1}, (7, 8): {"u": 4}, (7, 10): {"u": 2},
        (8, 10): {"u": 5}, (9, 10): {"u": 3},
    },
}
flow_source, flow_sink = 0, 10


def residual_reachable(network, flows, start, reverse=False, tol=1e-7):
    '''Find residual-reachable nodes; reverse=True finds nodes that can reach start.

    tol is an absolute tolerance in flow units. Supply a feasible flow.
    '''
    adjacency = {node: set() for node in network["nodes"]}
    for (i, j), data in network["edges"].items():
        if data["u"] - flows[i, j] > tol:
            adjacency[i].add(j)
        if flows[i, j] > tol:
            adjacency[j].add(i)
    if reverse:
        reversed_adjacency = {node: set() for node in network["nodes"]}
        for i, neighbors in adjacency.items():
            for j in neighbors:
                reversed_adjacency[j].add(i)
        adjacency = reversed_adjacency
    reached = {start}
    pending = [start]
    while pending:
        node = pending.pop()
        for neighbor in adjacency[node] - reached:
            reached.add(neighbor)
            pending.append(neighbor)
    return reached

In [ ]:
# def build_max_flow(network, source, sink):
#     # Use continuous nonnegative flows and throughput; replace every old balance.
#     # Add explicit capacity rows, maximize throughput, and attach a dual suffix.
#     return model
# Solve with HiGHS; report throughput, arc flows and capacity duals.

In [ ]:
# Call residual_reachable from the source, then in reverse from the sink.
# Complement the second set to obtain a source side.
# List arcs leaving each side and sum capacities from flow_network, without Pyomo.
# Explain the matching flow/cut certificate and compare with the capacity duals.

In [ ]:
# Predict all three changes before solving.
# Copy the original capacities for each experiment; do not accumulate changes.
# Rebuild and solve, then find a cut explaining the last throughput value.

In [ ]:
# State the directed-incidence integrality theorem with its data assumptions.
# Solve the triangle LP for both objective senses, retaining continuous variables.
# Prove the fractional maximum is unique and determine the best integral value.
# Explain what the integral minimizer does and does not establish.

## Exercise 5: the dual of a bounded allocation LP (optional)

Three divisible activities have rewards $c=(4,3,1)$ and resource requirements
$w=(1,2,1)$. Choose fractions $\xi_i$ subject to a resource budget $B\ge0$:

$$
\max\ c^\top\xi\quad\text{s.t.}\quad
w^\top\xi\le B,\qquad 0\le\xi_i\le1\quad(i=1,2,3).
$$

**(a)** Derive the dual before coding it. Use a multiplier $\lambda$ for the
budget and $\mu_i$ for each upper bound. Explain their signs, the dual
constraints and objective. Justify strong duality by feasibility and boundedness.

**(b)** Implement both LPs with builders that take $c,w,B$ as arguments. At
$B=2$, compare primal and dual feasibility, objective values and complementary
slackness. Keep the allocation variables continuous. Check: the value is 5.5.
Interpret $\lambda$ as the marginal value of additional resource near this budget.

**(c)** Repeat at $B=0$ and $B=\sum_iw_i=4$; the values are 0 and 8. Describe
all optimal dual multipliers at each endpoint. Compare feasibility and objective
values, allowing different solvers to select different optimal multipliers.

In [ ]:
allocation_c = {1: 4, 2: 3, 3: 1}
allocation_w = {1: 1, 2: 2, 3: 1}

In [ ]:
# Derive the dual by taking the supremum of the Lagrangian over xi>=0.
# def build_allocation_primal(c, w, B):
#     # Use continuous fractions, upper bounds and a weighted budget row.
#     return model
# def build_allocation_dual(c, w, B):
#     # Use nonnegative lambda and mu, with the derived constraints and objective.
#     return model

In [ ]:
# Solve both LPs at B=2, then at B=0 and B=4, using fresh models.
# Check feasibility, the objective gap and all three complementary-slackness products.
# Derive the endpoint families and check membership of each returned dual solution.

What did your checks establish, which errors could they detect, and what remains
unchecked?